In [0]:
from pyspark.sql.functions import * 
from pyspark.sql.types import * 
from delta.tables import DeltaTable

In [0]:
cust_address = spark.read.table("databricksintech.silver.customer_address")
cust_address.limit(3).display()

Cust_id,Add_id,Address_type,Modified_date
29485,1086,Main Office,2007-09-01
29486,621,Main Office,2005-09-01
29489,1069,Main Office,2005-07-01


In [0]:
address = spark.read.table("databricksintech.silver.address")
address.limit(3).display()

Address_id,Address_line1,Address_line2,City,State_province,Country_region,Postal_code,rowguid,Modified_date
9,8713 Yosemite Ct.,null,Bothell,Washington,United States,98011,268af621-76d7-4c78-9441-144fd139821a,2006-07-01
11,1318 Lasalle Street,null,Bothell,Washington,United States,98011,981b3303-aca2-49c7-9a96-fb670785b269,2007-04-01
25,9178 Jumping St.,null,Dallas,Texas,United States,75201,c8df3bd9-48f0-4654-a8dd-14a67a84d3c6,2006-09-01


In [0]:
address_joined = address.join(cust_address, address.Address_id==cust_address.Add_id, how='inner')
address_joined = address_joined.drop('rowguid', 'Modified_date', "Add_id")
address_joined.limit(5).display()

Address_id,Address_line1,Address_line2,City,State_province,Country_region,Postal_code,Cust_id,Address_type
9,8713 Yosemite Ct.,null,Bothell,Washington,United States,98011,30018,Shipping
11,1318 Lasalle Street,null,Bothell,Washington,United States,98011,29545,Shipping
25,9178 Jumping St.,null,Dallas,Texas,United States,75201,29954,Shipping
28,9228 Via Del Sol,null,Phoenix,Arizona,United States,85004,29897,Shipping
32,26910 Indela Road,null,Montreal,Quebec,Canada,H1Y 2H5,29503,Shipping


In [0]:
address_joined.columns

['Address_id',
 'Address_line1',
 'Address_line2',
 'City',
 'State_province',
 'Country_region',
 'Postal_code',
 'Cust_id',
 'Address_type']

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS databricksintech.gold;

CREATE TABLE IF NOT EXISTS databricksintech.gold.dim_customer_address
(
    Address_id STRING,
    Address_line1 STRING,
    Address_line2 STRING,
    City STRING,
    State_province STRING,
    Country_region STRING,
    Postal_code STRING,
    Cust_id STRING,
    Address_type STRING
    
) 
USING DELTA
LOCATION 'abfss://gold@intechaccstorage.dfs.core.windows.net/dim_customer_address';

In [0]:
# Reference target Delta table
gold_table = DeltaTable.forName(spark, "databricksintech.gold.dim_customer_address")

# Execute the Merge (Upsert)
gold_table.alias("target").merge(
    source = address_joined.alias("source"),
    condition = (
        "target.Address_id = source.Address_id"
        
    )
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

print("Done!")

Done!
